## Import Preprocessing

In [10]:
from preprocessing2 import (
    df_zscore, df_zscore_drop, df_zscore_replace,
    df_log, df_log_drop, df_log_replace,
    df_minmax, df_minmax_drop, df_minmax_replace,
    df_decimal, df_decimal_drop, df_decimal_replace,
)

print('Semua dataset berhasil diimport!')

Semua dataset berhasil diimport!


## Model Input

In [11]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd

def run_lgbm(df, dataset_name, test_size=0.2, random_state=42):
    df_clean = df.copy()
    
    # Drop ID kalau ada
    if 'Loan_ID' in df_clean.columns:
        df_clean = df_clean.drop(columns=['Loan_ID'])

    # Split fitur & target
    X = df_clean.drop(columns=['Loan_Status'])
    y = df_clean['Loan_Status']

    # Train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Model LightGBM
    lgbm = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=-1,
        random_state=random_state
    )

    # Training
    lgbm.fit(X_train, y_train)

    # Prediksi
    y_pred = lgbm.predict(X_test)

    return {
        'Dataset'   : dataset_name,
        'Accuracy'  : round(accuracy_score(y_test, y_pred), 4),
        'Precision' : round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall'    : round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1 Score'  : round(f1_score(y_test, y_pred, zero_division=0), 4),
        'y_test'    : y_test,
        'y_pred'    : y_pred,
        'cm'        : confusion_matrix(y_test, y_pred),
        'model'     : lgbm,
        'X_train'   : X_train,
        'X_test'    : X_test,
        'feature_names': X.columns.tolist(),
    }

print('Fungsi LightGBM siap!')

Fungsi LightGBM siap!


## Training Model

In [12]:
datasets = {
    'ZScore (Original)' : df_zscore,
    'ZScore (Drop)'     : df_zscore_drop,
    'ZScore (Replace)'  : df_zscore_replace,
    'Log (Original)'    : df_log,
    'Log (Drop)'        : df_log_drop,
    'Log (Replace)'     : df_log_replace,
    'MinMax (Original)' : df_minmax,
    'MinMax (Drop)'     : df_minmax_drop,
    'MinMax (Replace)'  : df_minmax_replace,
    'Decimal (Original)': df_decimal,
    'Decimal (Drop)'    : df_decimal_drop,
    'Decimal (Replace)' : df_decimal_replace,
}

results = {}
for name, data in datasets.items():
    print(f'Training: {name} ...')
    results[name] = run_lgbm(data, name)

print('\nSemua model selesai ditraining!')

Training: ZScore (Original) ...
[LightGBM] [Info] Number of positive: 337, number of negative: 154
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000298 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 544
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.686354 -> initscore=0.783130
[LightGBM] [Info] Start training from score 0.783130
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training: ZScore (Drop) ...
[LightGBM] [Info] Number of positive: 280, number of negative: 125
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000197 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 461
[LightGBM] [Info] Number of data points in the train set: 405, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.691358 -> initscore=0.806476
[LightGBM] [Info] Start training from score 0.806476
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

## Tabel Perbandingan

In [13]:
import subprocess
subprocess.run(["pip", "install", "jinja2", "--break-system-packages"])

Defaulting to user installation because normal site-packages is not writeable


CompletedProcess(args=['pip', 'install', 'jinja2', '--break-system-packages'], returncode=0)

In [14]:
summary = pd.DataFrame([
    {
        'Dataset'   : r['Dataset'],
        'Accuracy'  : r['Accuracy'],
        'Precision' : r['Precision'],
        'Recall'    : r['Recall'],
        'F1 Score'  : r['F1 Score'],
    }
    for r in results.values()
]).sort_values('F1 Score', ascending=False).reset_index(drop=True)
print(summary.to_string(index=False))

           Dataset  Accuracy  Precision  Recall  F1 Score
    Decimal (Drop)    0.8824     0.8718  0.9714    0.9189
        Log (Drop)    0.8824     0.8718  0.9714    0.9189
     ZScore (Drop)    0.8627     0.8684  0.9429    0.9041
     MinMax (Drop)    0.8627     0.8684  0.9429    0.9041
 MinMax (Original)    0.8537     0.8764  0.9176    0.8966
 ZScore (Original)    0.8374     0.8652  0.9059    0.8851
  ZScore (Replace)    0.8293     0.8636  0.8941    0.8786
    Log (Original)    0.8211     0.8462  0.9059    0.8750
Decimal (Original)    0.8211     0.8462  0.9059    0.8750
     Log (Replace)    0.8211     0.8621  0.8824    0.8721
  MinMax (Replace)    0.8211     0.8621  0.8824    0.8721
 Decimal (Replace)    0.8211     0.8621  0.8824    0.8721
